# Notebook khởi đầu

1. Bắt buộc dùng **link tài nguyên chỉ đọc của BTC** ở mục **Tài nguyên → Bộ dữ liệu**: dán link https vào `DATASET_URL`, notebook tự tải về, giải nén nếu là tệp `.zip` rồi đọc tệp Test và tệp Train bên trong.
2. Điền tên cột, `TEST_FILE`, `TRAIN_FILE` theo cuộc thi; tự hoàn thiện hai bước `TODO` (tiền xử lý, huấn luyện/dự đoán).
3. Chạy lại notebook để tạo `prediction.csv` rồi nộp CSV cùng notebook `.ipynb`.

In [ ]:
# 1. THƯ VIỆN VÀ SEED
import random
import shutil
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Seed thêm nếu môi trường có cài; thiếu thư viện nào thì bỏ qua thư viện đó.
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
except ImportError:
    pass

# sklearn không có seed toàn cục — truyền random_state=SEED khi tạo model.


In [ ]:
# 2. CẤU HÌNH — điền theo cuộc thi đang tham gia.
DATASET_URL = ""  # Bắt buộc: dán link "Bộ dữ liệu" của BTC (link tài nguyên chỉ đọc).
ID_COLUMN = "id"
PREDICTION_COLUMNS = ["prediction"]  # Tất cả cột dự đoán theo đúng thứ tự trong Hướng dẫn.
TEST_FILE = "test.csv"  # Tên tệp Test trong bộ dữ liệu BTC.
TRAIN_FILE = "train.csv"  # Tên tệp Train trong bộ dữ liệu BTC (đọc thử ở bước 3).
OUTPUT_FILE = "prediction.csv"

In [ ]:
# 3. TẢI VÀ ĐỌC DỮ LIỆU TỪ LINK TÀI NGUYÊN CỦA BTC
from urllib.parse import urlparse

assert DATASET_URL.startswith("https://"), "Dán link tài nguyên BTC (https://) vào DATASET_URL."
DATA_DIR = Path("dataset_btc")
DATA_DIR.mkdir(exist_ok=True)
archive = DATA_DIR / (Path(urlparse(DATASET_URL).path).name or "dataset")
# Nhiều host chặn User-Agent mặc định của Python bằng 403 nên gửi UA trình duyệt khi tải.
request = Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request, timeout=120) as response, open(archive, "wb") as file:
    shutil.copyfileobj(response, file)
if archive.suffix.lower() == ".zip" or zipfile.is_zipfile(archive):
    # Tệp không đuôi .zip đổi tên kèm đuôi trước khi giải nén để không trùng
    # thư mục/tệp bên trong zip (zip thật hay có một thư mục gốc cùng tên).
    if archive.suffix.lower() != ".zip":
        archive = archive.rename(archive.with_name(archive.name + ".zip"))
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(DATA_DIR)
        downloaded = [
            str(DATA_DIR / member.filename)
            for member in zip_file.infolist()
            if not member.is_dir()
        ]
    archive.unlink()
    print(f"Đã giải nén {len(downloaded)} tệp vào {DATA_DIR}/")
else:
    downloaded = [str(archive)]

test_files = [Path(path) for path in downloaded if Path(path).name == TEST_FILE]
assert len(test_files) == 1, f"Bộ dữ liệu BTC cần đúng một tệp {TEST_FILE}."
test_path = test_files[0]
test = pd.read_csv(test_path)
print("Đã đọc:", test_path, "| Kích thước Test:", test.shape)

# Đọc thử tệp Train để kiểm tra (bỏ qua nếu bộ dữ liệu chỉ có Test).
train_files = [Path(path) for path in downloaded if Path(path).name == TRAIN_FILE]
if train_files:
    train = pd.read_csv(train_files[0])
    print("Đã đọc:", train_files[0], "| Kích thước Train:", train.shape)
else:
    train = None
    print(f"Không thấy tệp {TRAIN_FILE} trong bộ dữ liệu BTC — bỏ qua đọc Train.")
test.head()


In [ ]:
# 4. KIỂM TRA CỘT ID
assert ID_COLUMN in test.columns, f"Thiếu cột {ID_COLUMN!r} trong {test_path}."
ids = test[ID_COLUMN]
assert ids.notna().all(), "Cột ID có giá trị rỗng."
assert not ids.duplicated().any(), "Cột ID bị trùng."


In [ ]:
# 5. TIỀN XỬ LÝ DỮ LIỆU — TODO: xử lý đặc trưng cho `train`/`test` đã đọc ở bước 3.


In [ ]:
# 6. HUẤN LUYỆN MÔ HÌNH — TODO: tạo predictions cho từng dòng test.
# predictions: DataFrame có các cột PREDICTION_COLUMNS theo đúng thứ tự, cùng số dòng với test.


In [ ]:
# 7. KÍCH THƯỚC MÔ HÌNH — gọi sau khi tạo `model` ở bước 6 (PyTorch hoặc TensorFlow/Keras).
def model_size(model):
    """In số tham số và dung lượng xấp xỉ (float32) của mô hình."""
    if hasattr(model, "count_params"):  # TensorFlow/Keras
        total = int(model.count_params())
    else:  # PyTorch
        total = sum(p.numel() for p in model.parameters())
    print(f"{total:,} tham số (~{total * 4 / 1024 ** 2:.1f} MB float32)")
    return total

# model_size(model)  # Bỏ comment sau khi tạo model ở bước 6.


In [ ]:
# 8. GHÉP DỰ ĐOÁN VỚI ID (chạy sau khi đã hoàn thiện TODO)
assert PREDICTION_COLUMNS and ID_COLUMN not in PREDICTION_COLUMNS
assert list(predictions.columns) == PREDICTION_COLUMNS
assert len(predictions) == len(test)
submission = pd.concat([test[[ID_COLUMN]].reset_index(drop=True), predictions.reset_index(drop=True)], axis=1)


In [ ]:
# 9. XUẤT CSV — đúng tên và thứ tự cột cuộc thi yêu cầu.
assert list(submission.columns) == [ID_COLUMN, *PREDICTION_COLUMNS]
assert len(submission) == len(test)
submission.to_csv(OUTPUT_FILE, index=False)
written = pd.read_csv(OUTPUT_FILE)
assert list(written.columns) == [ID_COLUMN, *PREDICTION_COLUMNS]
assert len(written) == len(test)
print(f"Đã ghi {OUTPUT_FILE}: {len(written)} dòng.")